In [0]:
%sql
-- Setup catalog if not exists

-- Setup schemas for medallion architecture, you can also use the GUI
-- Schema == Database
CREATE SCHEMA IF NOT EXISTS jarvis_training.bronze;
CREATE SCHEMA IF NOT EXISTS jarvis_training.silver;
CREATE SCHEMA IF NOT EXISTS jarvis_training.gold;
CREATE VOLUME IF NOT EXISTS jarvis_training.bronze.stock_volume;

In [0]:
import time
import requests
import json
from datetime import datetime
from pyspark.sql.functions import to_date

In [0]:
import time
import requests
import json
from datetime import datetime

url = "https://www.alphavantage.co/query"
symbols = ["AAPL", "GOOGL", "META", "IBM"]
all_counts = []

for i, symbol in enumerate(symbols):    
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey": "UPIWHXKF9JUMO257",
        "outputsize": "compact"
    }
    
    response = requests.get(url, params=querystring)
    data = response.json()
    rows = []
    ts = data["Time Series (Daily)"]
    
    for date, values in ts.items():
        row = {
            "company": symbol,
            "date": date,
            "close": float(values["4. close"]),
            "open": float(values["1. open"]),
            "high": float(values["2. high"]),
            "low": float(values["3. low"]),
            "volume": int(values["5. volume"])
        }
        rows.append(row)
    
    target_dir = "/Volumes/jarvis_training/bronze/stock_volume"
    dbutils.fs.mkdirs(target_dir)
    file_name = f"{symbol.lower()}.json"
    file_path = f"{target_dir}/{file_name}"
    
    json_lines = "\n".join(json.dumps(row) for row in rows)
    dbutils.fs.put(file_path, json_lines, overwrite=True)
    all_counts.append((symbol, len(rows), file_path))
    
    if i < len(symbols) - 1:
        time.sleep(5)


Wrote 12595 bytes.
Wrote 12717 bytes.
Wrote 12572 bytes.
Wrote 12402 bytes.
